<a href="https://colab.research.google.com/github/abhayjha800/ML_Projects/blob/main/cat_v_dog_pytorch.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/

In [ ]:
!kaggle datasets download shaunthesheep/microsoft-catsvsdogs-dataset

Dataset URL: https://www.kaggle.com/datasets/shaunthesheep/microsoft-catsvsdogs-dataset
License(s): other
 93% 734M/788M [00:04<00:01, 43.1MB/s]
100% 788M/788M [00:04<00:00, 199MB/s] 


In [ ]:
import zipfile
zip_ref = zipfile.ZipFile('microsoft-catsvsdogs-dataset.zip', 'r')
zip_ref.extractall('/content')
zip_ref.close()

In [24]:
import torch
from torch.utils.data import Dataset, DataLoader, random_split
import torch.nn as nn
import torch.optim as optim
from torchvision import datasets, transforms
import torch.nn.functional as f

In [25]:
device = "cuda" if torch.cuda.is_available() else "cpu"
print(device)

cuda


In [26]:
transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.5,0.5,0.5],
                         std=[0.5,0.5,0.5])
                                ])

In [27]:
from PIL import Image
from torchvision.datasets import ImageFolder

class SafeImageFolder(ImageFolder):
    def __getitem__(self, index):
        path, target = self.samples[index]
        try:
            # Try to open the image normally
            sample = self.loader(path)
        except Exception:
            # If fails, return a black image instead
            sample = Image.new('RGB', (224, 224))
        if self.transform is not None:
            sample = self.transform(sample)
        if self.target_transform is not None:
            target = self.target_transform(target)
        return sample, target



full_dataset = SafeImageFolder(root="PetImages", transform=transform)
print(f"Total images: {len(full_dataset)}")
print(f"Classes: {full_dataset.classes}")  # should show ['cats', 'dogs']


Total images: 25000
Classes: ['Cat', 'Dog']


In [28]:
train_size = int(0.8 * len(full_dataset))
test_size = len(full_dataset) - train_size

train_dataset, test_dataset = random_split(full_dataset, [train_size, test_size])

In [29]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32)

In [30]:
class SimpleCNN(nn.Module):
  def __init__(self):
    super(SimpleCNN, self).__init__()
    self.network = nn.Sequential(
        nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
        nn.ReLU(),
        nn.MaxPool2d(2,2),

        nn.Flatten(),

        nn.Linear(128*28*28, 64),
        nn.ReLU(),
        nn.Dropout(0.3),
        nn.Linear(64,2)

    )

  def forward(self, x):
    return self.network(x)

In [31]:
model = SimpleCNN().to(device)

In [ ]:
!pip install torchinfo

In [32]:
from torchinfo import summary
summary(model, input_size=(1,3,224,224))

Layer (type:depth-idx)                   Output Shape              Param #
SimpleCNN                                [1, 2]                    --
├─Sequential: 1-1                        [1, 2]                    --
│    └─Conv2d: 2-1                       [1, 32, 224, 224]         896
│    └─ReLU: 2-2                         [1, 32, 224, 224]         --
│    └─MaxPool2d: 2-3                    [1, 32, 112, 112]         --
│    └─Conv2d: 2-4                       [1, 64, 112, 112]         18,496
│    └─ReLU: 2-5                         [1, 64, 112, 112]         --
│    └─MaxPool2d: 2-6                    [1, 64, 56, 56]           --
│    └─Conv2d: 2-7                       [1, 128, 56, 56]          73,856
│    └─ReLU: 2-8                         [1, 128, 56, 56]          --
│    └─MaxPool2d: 2-9                    [1, 128, 28, 28]          --
│    └─Flatten: 2-10                     [1, 100352]               --
│    └─Linear: 2-11                      [1, 64]                   6,422,592

In [33]:
criterion  = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
num_epochs = 5

In [34]:
for epoch in range(num_epochs):
  running_loss = 0.0
  model.train()

  for images, labels in train_loader:
    try:
      images, labels = images.to(device), labels.to(device)

      optimizer.zero_grad()

      outputs = model(images)
      loss = criterion(outputs, labels)

      loss.backward()
      optimizer.step()

      running_loss += loss.item()
    except UnidentifiedImageError:
      print(f"Skipping corrupted image in batch.")
      continue


  epoch_loss = running_loss / len(train_loader)
  print(f"Epoch {epoch+1}/{num_epochs}, Loss: {epoch_loss:.4f}")

Epoch 1/5, Loss: 0.6244
Epoch 2/5, Loss: 0.5056
Epoch 3/5, Loss: 0.4184
Epoch 4/5, Loss: 0.3417
Epoch 5/5, Loss: 0.2561


In [35]:
model.eval()
correct = 0
total = 0

with torch.no_grad():
  for images, labels in test_loader:
    images, labels = images.to(device), labels.to(device)
    outputs = model(images)
    _, predicted = torch.max(outputs, 1)
    total += labels.size(0)
    correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f"Test Accuracy: {accuracy:.2f}%")


Test Accuracy: 81.98%
